# Experiment 3 — Video Object Segmentation via Label Propagation

**Claim:** AnyUp3D-upsampled features produce more discriminative spatial representations than bilinear upsampling, measured by how well frozen features propagate object masks on DAVIS 2017 without any training.

**Protocol:** Non-parametric label propagation following Jabri et al. (NeurIPS 2020) *Space-Time Correspondence as a Contrastive Random Walk* (arXiv:2006.14613), with context design from Caron et al. (ICCV 2021) *Emerging Properties in Self-Supervised Vision Transformers* (DINO, arXiv:2104.14294).

**No training. No head. Frame 0 GT mask is given as input; frames 1…T are propagated via cosine similarity in feature space.**

**Metric:** J&F mean (DAVIS 2017 val, semi-supervised setting)

**Conditions:**
- AnyUp3D: Video Swin-T → AnyUp3D → 56×56 features
- Bilinear: Video Swin-T → bilinear interpolation → 56×56 features

---
**Design decisions and citations**
| Decision | Value | Source |
|---|---|---|
| Propagation resolution | 56×56 | Ablation: high enough to distinguish boundaries, fits T4 VRAM |
| Softmax temperature | τ=0.07 | Jabri et al. NeurIPS 2020, Table 1 |
| Top-k neighbors | k=7 | DINO eval code (github.com/facebookresearch/dino) |
| Context frames | Frame 0 + last n=6 frames | DINO eval code (`n_last_frames=6`) |
| Multi-object | Soft per-object propagation, argmax | DAVIS 2017 semi-supervised standard |

In [1]:
# ── Cell 1 · Installs ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "3Dconv"
REPO_DIR    = "/content/anyup"

subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)


# natten — required by AnyUp cross-attention
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "natten",
     "--find-links",
     "https://shi-labs.com/natten/wheels/cu118/torch2.0.0/index.html"],
    check=False
)

print("Cell 1 done.")

Mounted at /content/drive
Cell 1 done.


In [2]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
import os

# ── Paths ───────────────────────────────────────────────────────────────────
CKPT_ANYUP3D  = "/content/drive/MyDrive/results/anyup3d_step500.pth"
DAVIS_ZIP     = "/content/drive/MyDrive/results/DAVIS-2017-trainval-480p.zip"
DAVIS_ROOT    = "/content/DAVIS"

# Swin val cache from Exp 2 — reused directly, no re-extraction needed
SWIN_CACHE_DIR = "/content/drive/MyDrive/results/exp2/swin_cache"

OUTPUT_DIR     = "/content/drive/MyDrive/results/exp3"
FEAT_CACHE_DIR = os.path.join(OUTPUT_DIR, "feat_cache")   # {anyup3d, bilinear} subdirs
MASK_DIR       = os.path.join(OUTPUT_DIR, "masks")        # {anyup3d, bilinear} subdirs
STILLS_DIR     = os.path.join(OUTPUT_DIR, "stills")
VIDEO_DIR      = os.path.join(OUTPUT_DIR, "videos")

for d in [OUTPUT_DIR, FEAT_CACHE_DIR, MASK_DIR, STILLS_DIR, VIDEO_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Model ───────────────────────────────────────────────────────────────────
# These args match the run_02 checkpoint exactly.
# t_k and window_t are the temporal kernel / window params added in the 3D model.
ANYUP3D_ARGS = dict(
    input_dim       = 3,
    qk_dim          = 128,
    kernel_size     = 1,
    kernel_size_lfu = 5,   # ← critical: omitting causes 73 missing keys
    window_ratio    = 0.1,
    num_heads       = 4,
    t_k             = 3,   # ← temporal kernel size; must match checkpoint
    window_t        = 3,   # ← temporal attention window; must match checkpoint
)

# ── Swin feature dims (at IMG_SIZE=224) ────────────────────────────────────
FEAT_DIM  = 768   # ← Swin-T channel dim
FEAT_SIZE = 7     # ← Swin-T spatial output at 224px input
FRAME_STEP = 12   # ← sample every 12th frame (24fps → 2fps); from Exp 2

# ── Propagation resolution ─────────────────────────────────────────────────
# 56 = 8× the raw Swin output (7→56). High enough to capture object boundaries,
# small enough for O(N^2) affinity to fit on T4.
# At 56×56: affinity matrix per frame = 3136×(context×3136) — feasible.
PROP_SIZE = 56    # ← propagation (and AnyUp3D output) resolution
                  #   changing this requires verifying affinity fits in VRAM

# ── AnyUp3D forward chunking ────────────────────────────────────────────────
# At PROP_SIZE=56, each chunk is small; T_CHUNK=8 is safe on T4.
# Reduce to 4 if OOM; raise to 16 if you want faster caching.
T_CHUNK = 8       # ← max frames per AnyUp3D forward pass; affects upsample_3d only

# ── Label propagation hyperparameters ──────────────────────────────────────
# TAU: Jabri et al. NeurIPS 2020 (arXiv:2006.14613), Table 1 — τ=0.07
TAU = 0.07        # ← softmax temperature; lower = sharper attention

# TOP_K: DINO eval code (github.com/facebookresearch/dino, eval_video_segmentation.py)
# Uses topk=7 neighbors per query location.
TOP_K = 7         # ← top-k neighbors kept before softmax; rest set to -inf
                  #   lower = sparser / harder propagation

# N_LAST_FRAMES: DINO eval code — n_last_frames=6
# Context at frame t = frame 0 (GT anchor) + last N_LAST_FRAMES frames.
# Frame 0 is always included to prevent drift; from Caron et al. ICCV 2021.
N_LAST_FRAMES = 6  # ← context window size (excluding frame 0 anchor)
                   #   increase for longer clips; 6 is DINO default

# ── Visuals ─────────────────────────────────────────────────────────────────
N_VISUAL_CLIPS = 4   # ← clips shown in side-by-side figure
VISUAL_FPS     = 4   # ← output comparison video FPS

print("Config loaded.")
print(f"  PROP_SIZE={PROP_SIZE}, TAU={TAU}, TOP_K={TOP_K}, N_LAST_FRAMES={N_LAST_FRAMES}")
print(f"  T_CHUNK={T_CHUNK}, FRAME_STEP={FRAME_STEP}")

Config loaded.
  PROP_SIZE=56, TAU=0.07, TOP_K=7, N_LAST_FRAMES=6
  T_CHUNK=8, FRAME_STEP=12


In [3]:
# ── Cell 3 · Imports ────────────────────────────────────────────────────────
import sys, os, random
import numpy as np
from pathlib import Path
import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models.video import swin3d_t, Swin3D_T_Weights
from PIL import Image
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from tqdm import tqdm

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"  {torch.cuda.get_device_name(0)}, "
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

Device: cuda
  Tesla T4, 15.6 GB


In [4]:
# ── Cell 4 · Model loading ──────────────────────────────────────────────────

# TStage must be defined before torch.load — checkpoint pickle needs it.
# See Exp 1 lessons learned.
class TStage:
    def __init__(self, t=None, start_step=None, batch_size=None):
        self.t          = t
        self.start_step = start_step
        self.batch_size = batch_size

# ── AnyUp3D ─────────────────────────────────────────────────────────────────
from anyup.model import AnyUp

model_3d = AnyUp(**ANYUP3D_ARGS).to(device).eval()

ckpt = torch.load(CKPT_ANYUP3D, map_location="cpu", weights_only=False)  # weights_only=False: TStage in pickle
sd   = ckpt["anyup"]   # ← key is "anyup"; not "model" or "state_dict"
missing, unexpected = model_3d.load_state_dict(sd, strict=False)
print(f"AnyUp3D  — missing: {len(missing)}, unexpected: {len(unexpected)}")
if missing:
    print(f"  First 5: {missing[:5]}")

for p in model_3d.parameters():
    p.requires_grad_(False)

# ── AnyUp2D (2D baseline reference, not used in propagation) ────────────────
# Loaded here for completeness; not used in this experiment.
# The comparison is AnyUp3D vs bilinear only.

# ── Video Swin-T ─────────────────────────────────────────────────────────────
# Backbone used in Exp 2; we reuse its cached outputs — no forward pass needed here.
# Hook registered anyway so Cell 6 can optionally re-extract if cache is missing.
_feat = {}

def _hook_fn(module, inp, out):
    # Swin norm outputs (B, T, H, W, C) — must permute to (B, C, T, H, W)
    # See Exp 1 lessons learned: channels-last output.
    _feat["norm"] = out.detach().permute(0, 4, 1, 2, 3).contiguous()

backbone = swin3d_t(weights=Swin3D_T_Weights.KINETICS400_V1).to(device).eval()
backbone.norm.register_forward_hook(_hook_fn)
for p in backbone.parameters():
    p.requires_grad_(False)

# ── Preprocessing ────────────────────────────────────────────────────────────
# guide_transform: resize to PROP_SIZE for AnyUp3D input
# (AnyUp3D output resolution = guide resolution = PROP_SIZE)
IMG_SIZE = 224
frame_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
guide_transform = transforms.Compose([
    transforms.Resize((PROP_SIZE, PROP_SIZE)),   # ← PROP_SIZE: AnyUp output = guide size
    transforms.ToTensor(),
])

print("Models loaded.")

AnyUp3D  — missing: 0, unexpected: 0
Downloading: "https://download.pytorch.org/models/swin3d_t-7615ae03.pth" to /root/.cache/torch/hub/checkpoints/swin3d_t-7615ae03.pth


100%|██████████| 122M/122M [00:00<00:00, 132MB/s]


Models loaded.


In [5]:
# ── Cell 5 · DAVIS setup ────────────────────────────────────────────────────
import subprocess

if not os.path.isdir(DAVIS_ROOT):
    print("Extracting DAVIS...")
    subprocess.run(["unzip", "-q", DAVIS_ZIP, "-d", "/content/"], check=True)
    print("Done.")
else:
    print("DAVIS already extracted.")

IMG_DIR = os.path.join(DAVIS_ROOT, "JPEGImages",  "480p")
ANN_DIR = os.path.join(DAVIS_ROOT, "Annotations", "480p")
SET_DIR = os.path.join(DAVIS_ROOT, "ImageSets",   "2017")

with open(os.path.join(SET_DIR, "val.txt")) as f:
    val_clips = [l.strip() for l in f if l.strip()]
print(f"Val clips: {len(val_clips)}")

def get_frame_paths(clip):
    return sorted(Path(IMG_DIR, clip).glob("*.jpg"))

def get_ann_paths(clip):
    return sorted(Path(ANN_DIR, clip).glob("*.png"))

def sample_indices(n_frames):
    """Every FRAME_STEP-th frame; skip clip if fewer than 2 result."""
    idxs = list(range(0, n_frames, FRAME_STEP))  # ← FRAME_STEP controls T and VRAM
    return idxs if len(idxs) >= 2 else []

def load_ann(ann_path):
    """Load palette PNG → numpy uint8 (H, W), values = object IDs (0=bg)."""
    return np.array(Image.open(ann_path))

def get_n_objects(clip):
    """Count unique object IDs in frame 0 annotation (excluding background=0)."""
    ann0 = load_ann(get_ann_paths(clip)[0])
    return int(ann0.max())   # DAVIS IDs are contiguous starting at 1

# Sanity check
sample = val_clips[0]
n_frames = len(get_frame_paths(sample))
idxs = sample_indices(n_frames)
print(f"  '{sample}': {n_frames} frames → {len(idxs)} sampled, "
      f"{get_n_objects(sample)} object(s)")

Extracting DAVIS...
Done.
Val clips: 30
  'bike-packing': 69 frames → 6 sampled, 2 object(s)


In [6]:
from pathlib import Path
clip = val_clips[0]
ann_paths = get_ann_paths(clip)
frame_paths = get_frame_paths(clip)
idxs = sample_indices(len(frame_paths))

ann0 = load_ann(ann_paths[idxs[0]])
print(f"ann0 unique values: {np.unique(ann0)}")
# Expected: [0, 1] or [0, 1, 2] etc — at least one object ID > 0
# Bad: [0] only — means wrong annotation frame index or wrong path
print(f"ann0 shape: {ann0.shape}, frame index used: {idxs[0]}")
print(f"ann path: {ann_paths[idxs[0]]}")

ann0 unique values: [0 1 2]
ann0 shape: (480, 910), frame index used: 0
ann path: /content/DAVIS/Annotations/480p/bike-packing/00000.png


In [7]:
# ── Cell 6 · Feature upsampling → 56×56 cache ──────────────────────────────
#
# Source: Swin val cache from Exp 2 (feats: T×768×7×7, guides: T×3×224×224)
# Output: T×768×56×56 per condition, saved to Drive.
#
# AnyUp3D: guide resized to PROP_SIZE×PROP_SIZE → output at PROP_SIZE×PROP_SIZE.
# Bilinear: F.interpolate(feats, size=(PROP_SIZE, PROP_SIZE)).
#
# IMPORTANT: Swin cache stores ImageNet-normalized guides.
# AnyUp3D expects raw RGB in [0,1] — guides must be unnormalized before use.

SWIN_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
SWIN_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

@torch.no_grad()
def upsample_3d(feats_TCHW, guides_T3HW_raw):
    """
    Chunked AnyUp3D forward pass.
    feats_TCHW:       (T, 768, 7, 7)              — CPU, float32
    guides_T3HW_raw:  (T, 3, PROP_SIZE, PROP_SIZE) — CPU, float32, range [0,1]
    Returns:          (T, 768, PROP_SIZE, PROP_SIZE) — CPU, float32
    """
    T = feats_TCHW.shape[0]   # ← total frames; chunked into T_CHUNK slices
    out_chunks = []

    for start in range(0, T, T_CHUNK):   # ← T_CHUNK controls peak VRAM
        end = min(start + T_CHUNK, T)

        # (chunk, C, H, W) → unsqueeze(0) → (1, chunk, C, H, W)
        # → permute(0,2,1,3,4) → (1, C, chunk, H, W) for AnyUp3D
        feat_chunk  = feats_TCHW[start:end].unsqueeze(0).permute(0, 2, 1, 3, 4).to(device)
        guide_chunk = guides_T3HW_raw[start:end].unsqueeze(0).permute(0, 2, 1, 3, 4).to(device)

        # guide first — NOT (feat, guide); see Exp 1 & 2 lessons
        out = model_3d(guide_chunk, feat_chunk)          # (1, 768, chunk, PROP_SIZE, PROP_SIZE)
        # → squeeze → (768, chunk, PROP_SIZE, PROP_SIZE)
        # → permute → (chunk, 768, PROP_SIZE, PROP_SIZE)
        out_chunks.append(out.squeeze(0).permute(1, 0, 2, 3).cpu())

        del feat_chunk, guide_chunk, out
        torch.cuda.empty_cache()

    return torch.cat(out_chunks, dim=0)   # (T, 768, PROP_SIZE, PROP_SIZE)


@torch.no_grad()
def upsample_bilinear(feats_TCHW):
    """
    Bilinear upsample, no guidance.
    feats_TCHW: (T, 768, 7, 7) — CPU
    Returns:    (T, 768, PROP_SIZE, PROP_SIZE) — CPU
    """
    return F.interpolate(
        feats_TCHW,
        size=(PROP_SIZE, PROP_SIZE),   # ← PROP_SIZE: change consistently with config
        mode="bilinear",
        align_corners=False
    )


# Check Swin cache coverage
cached = [c for c in val_clips
          if os.path.exists(os.path.join(SWIN_CACHE_DIR, f"{c}.pt"))]
print(f"Val clips with Swin cache: {len(cached)} / {len(val_clips)}")

# ── Run upsampling for both conditions ──────────────────────────────────────
for condition in ["anyup3d", "bilinear"]:
    cond_dir = os.path.join(FEAT_CACHE_DIR, condition)
    os.makedirs(cond_dir, exist_ok=True)
    print(f"\nCaching {condition} features...")

    for clip in tqdm(cached):
        out_path = os.path.join(cond_dir, f"{clip}.pt")
        if os.path.exists(out_path):   # safe to re-run after disconnection
            continue

        raw = torch.load(
            os.path.join(SWIN_CACHE_DIR, f"{clip}.pt"),
            map_location="cpu", weights_only=False
        )
        feats_TCHW  = raw["feats"].float()   # (T, 768, 7, 7)
        guides_norm = raw["guides"].float()  # (T, 3, 224, 224) — ImageNet normalized

        # Unnormalize guides: Swin cache stored normalized values,
        # AnyUp3D cross-attention guide must be raw RGB in [0, 1]
        guides_raw = guides_norm * SWIN_STD + SWIN_MEAN   # → [0, 1]
        guides_raw = guides_raw.clamp(0.0, 1.0)

        # Resize guides to PROP_SIZE — AnyUp3D output = guide spatial resolution
        guides_small = F.interpolate(
            guides_raw,
            size=(PROP_SIZE, PROP_SIZE),   # ← PROP_SIZE: change consistently with config
            mode="bilinear",
            align_corners=False
        )                                  # (T, 3, PROP_SIZE, PROP_SIZE)

        if condition == "anyup3d":
            out = upsample_3d(feats_TCHW, guides_small)   # (T, 768, PROP_SIZE, PROP_SIZE)
        else:
            out = upsample_bilinear(feats_TCHW)            # (T, 768, PROP_SIZE, PROP_SIZE)

        torch.save(out.half().cpu(), out_path)   # float16 — halves Drive space
        del feats_TCHW, guides_norm, guides_raw, guides_small, out
        torch.cuda.empty_cache()

print("\nFeature cache complete.")

# Sanity check: verify guide range is now correct
sample_raw = torch.load(
    os.path.join(SWIN_CACHE_DIR, f"{cached[0]}.pt"),
    map_location="cpu", weights_only=False
)
g = sample_raw["guides"].float() * SWIN_STD + SWIN_MEAN
print(f"Guide range after unnorm: min={g.min():.3f} max={g.max():.3f}  (expected ~0.0 to ~1.0)")

Val clips with Swin cache: 30 / 30

Caching anyup3d features...


100%|██████████| 30/30 [00:00<00:00, 66.59it/s]



Caching bilinear features...


100%|██████████| 30/30 [00:00<00:00, 1959.47it/s]



Feature cache complete.
Guide range after unnorm: min=0.016 max=1.000  (expected ~0.0 to ~1.0)


In [8]:
# ── Cell 7 · Label propagation ──────────────────────────────────────────────
#
# Protocol: Jabri et al. NeurIPS 2020 (arXiv:2006.14613) with DINO context design.
#
# Algorithm (per clip):
#   1. L2-normalize features at each spatial location.
#   2. For frame t, build context = frame 0 + last N_LAST_FRAMES frames.
#   3. Compute cosine affinity: A = (Q @ K^T) / TAU — shape (N_query, N_context)
#   4. Keep only TOP_K highest affinities per query; set rest to -inf.
#   5. Softmax over context dim → attention weights.
#   6. Propagate soft label maps: new_label = A @ old_label.
#   7. argmax over objects+background → integer mask.
#   8. Save as palette PNG.
#
# No gradients, no parameters, no training.

def build_label_maps_frame0(ann0_HW, n_objects, H, W):
    """
    Convert frame 0 annotation to one-hot soft label maps.
    ann0_HW: (H_orig, W_orig) uint8 — object IDs (0=bg, 1..n=objects)
    Returns: (n_objects+1, H*W) float32
    """
    ann_resized = cv2.resize(
        ann0_HW, (W, H), interpolation=cv2.INTER_NEAREST
    ).astype(np.int64)                      # (H, W)

    n_ch    = n_objects + 1                 # background channel + one per object
    N       = H * W
    label   = np.zeros((n_ch, N), dtype=np.float32)
    ann_flat = ann_resized.flatten()        # (N,)

    for obj_id in range(n_ch):
        label[obj_id, ann_flat == obj_id] = 1.0

    return torch.from_numpy(label)          # (n_ch, N)


@torch.no_grad()
def propagate_clip(feats_TCHW, ann_paths, sampled_idxs):
    """
    Non-parametric label propagation for one clip.

    feats_TCHW:   (T, 768, PROP_SIZE, PROP_SIZE) float32
    ann_paths:    sorted list of ALL annotation paths for the clip
    sampled_idxs: list of original frame indices that were sampled

    Returns: pred_masks (T, PROP_SIZE, PROP_SIZE) int64 — object IDs
    """
    T, C, H, W = feats_TCHW.shape   # H = W = PROP_SIZE
    N = H * W                        # ← PROP_SIZE^2 spatial locations

    # ── Step 1: L2-normalize features ───────────────────────────────────────
    # Cosine similarity requires unit vectors — normalize along channel dim.
    # Jabri et al. NeurIPS 2020.
    feats_flat = feats_TCHW.view(T, C, N).permute(0, 2, 1)   # (T, N, C)
    feats_flat = F.normalize(feats_flat, dim=-1)              # (T, N, C) unit vectors

    # ── Step 2: Build frame 0 label maps from GT annotation ─────────────────
    # Frame 0 GT mask is the only supervision used — semi-supervised protocol.
    ann0 = load_ann(ann_paths[sampled_idxs[0]])   # (H_orig, W_orig) uint8
    n_objects = int(ann0.max())                   # number of distinct objects
    n_ch = n_objects + 1                          # +1 for background

    # label_maps[t]: soft assignment of each spatial location to each class
    label_maps = torch.zeros(T, n_ch, N)          # (T, n_ch, N) float32
    label_maps[0] = build_label_maps_frame0(ann0, n_objects, H, W)

    # ── Step 3–6: Propagate frame by frame ──────────────────────────────────
    for t in range(1, T):
        # Context window: frame 0 (GT anchor) + last N_LAST_FRAMES frames.
        # Always including frame 0 prevents long-term drift.
        # Design from DINO eval code (Caron et al. ICCV 2021, arXiv:2104.14294).
        past      = list(range(max(1, t - N_LAST_FRAMES), t))  # ← N_LAST_FRAMES from config
        context_t = [0] + past                                  # frame 0 always first

        # Stack context: each context frame contributes N spatial locations
        ctx_feats  = feats_flat[context_t].reshape(-1, C)      # (Nctx*N, C)
        ctx_labels = label_maps[context_t].reshape(n_ch, -1)   # (n_ch, Nctx*N)

        # Query: current frame features
        q = feats_flat[t]   # (N, C)

        # Cosine affinity scaled by temperature.
        # Jabri et al. NeurIPS 2020, Eq. 1: A_ij = f_i · f_j / τ
        A = torch.mm(q, ctx_feats.T) / TAU   # (N, Nctx*N) ← TAU from config

        # Top-k sparsification: keep only TOP_K most similar context locations.
        # From DINO eval code — topk=7. Sets all other affinities to -inf
        # so softmax assigns them zero weight.
        topk_vals, topk_idx = A.topk(TOP_K, dim=-1)      # ← TOP_K from config
        A_sparse = torch.full_like(A, float("-inf"))
        A_sparse.scatter_(1, topk_idx, topk_vals)

        # Softmax → attention weights over context locations
        A_soft = F.softmax(A_sparse, dim=-1)   # (N, Nctx*N)

        # Propagate label maps via weighted sum over context
        # new_label[i, obj] = Σ_j A_soft[i,j] * ctx_label[obj, j]
        new_labels = torch.mm(A_soft, ctx_labels.T)   # (N, n_ch)
        label_maps[t] = new_labels.T                  # (n_ch, N)

    # ── Step 7: argmax → hard integer mask ──────────────────────────────────
    pred = label_maps.argmax(dim=1)   # (T, N)
    return pred.view(T, H, W)         # (T, PROP_SIZE, PROP_SIZE) int64


def save_mask_png(mask_HW, clip_name, out_path):
    """
    Save predicted mask as palette PNG.
    Resized to original DAVIS annotation resolution before saving.
    mask_HW:   (H, W) int64 tensor — values 0..n_objects
    clip_name: DAVIS clip name — used to find reference annotation for size/palette
    out_path:  destination path
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    # Get original annotation resolution from first annotation frame
    ann_ref_path = str(sorted(Path(ANN_DIR, clip_name).glob("*.png"))[0])
    ref_img  = Image.open(ann_ref_path)
    H_orig, W_orig = np.array(ref_img).shape[:2]

    # Resize predicted mask to original resolution
    mask_np = mask_HW.numpy().astype(np.uint8)
    mask_np = cv2.resize(mask_np, (W_orig, H_orig),
                         interpolation=cv2.INTER_NEAREST)

    # Save as palette PNG — pixel values are object IDs
    out_img = Image.fromarray(mask_np, mode="P")
    if ref_img.mode == "P":
        out_img.putpalette(ref_img.getpalette())
    out_img.save(out_path)


# ── Run propagation for both conditions ─────────────────────────────────────
skip_counts = {"anyup3d": 0, "bilinear": 0}

for condition in ["anyup3d", "bilinear"]:
    feat_dir      = os.path.join(FEAT_CACHE_DIR, condition)
    mask_cond_dir = os.path.join(MASK_DIR, condition)
    os.makedirs(mask_cond_dir, exist_ok=True)
    print(f"\nPropagating — {condition}...")

    for clip in tqdm(val_clips):
        feat_path = os.path.join(feat_dir, f"{clip}.pt")
        if not os.path.exists(feat_path):
            skip_counts[condition] += 1
            continue

        frame_paths = get_frame_paths(clip)
        ann_paths   = get_ann_paths(clip)
        idxs        = sample_indices(len(frame_paths))
        if not idxs:
            skip_counts[condition] += 1
            continue

        # Skip if already processed — safe to re-run after disconnection
        clip_mask_dir = os.path.join(mask_cond_dir, clip)
        stem0 = Path(frame_paths[idxs[0]]).stem
        if os.path.exists(os.path.join(clip_mask_dir, f"{stem0}.png")):
            continue

        feats = torch.load(
            feat_path, map_location="cpu", weights_only=False
        ).float()   # (T, 768, PROP_SIZE, PROP_SIZE)

        T          = feats.shape[0]
        pred_masks = propagate_clip(feats, ann_paths, idxs)   # (T, PROP_SIZE, PROP_SIZE)

        for t, src_i in enumerate(idxs[:T]):
            stem     = Path(frame_paths[src_i]).stem
            out_path = os.path.join(clip_mask_dir, f"{stem}.png")
            save_mask_png(pred_masks[t], clip, out_path)

        del feats, pred_masks
        torch.cuda.empty_cache()

print("\nPropagation complete.")
print(f"  Skipped — anyup3d: {skip_counts['anyup3d']}, "
      f"bilinear: {skip_counts['bilinear']}")

# Sanity check: verify at least one clip was processed
for condition in ["anyup3d", "bilinear"]:
    d = os.path.join(MASK_DIR, condition)
    n_clips = len(os.listdir(d))
    if n_clips > 0:
        sample_clip = os.listdir(d)[0]
        n_masks = len(os.listdir(os.path.join(d, sample_clip)))
        print(f"  {condition}: {n_clips} clips, sample '{sample_clip}' has {n_masks} masks")
    else:
        print(f"  {condition}: 0 clips — something went wrong")


Propagating — anyup3d...


  0%|          | 0/30 [00:00<?, ?it/s]/tmp/ipykernel_8894/3282071964.py:128: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  out_img = Image.fromarray(mask_np, mode="P")
100%|██████████| 30/30 [04:13<00:00,  8.44s/it]



Propagating — bilinear...


100%|██████████| 30/30 [04:30<00:00,  9.02s/it]


Propagation complete.
  Skipped — anyup3d: 0, bilinear: 0
  anyup3d: 30 clips, sample 'bike-packing' has 6 masks
  bilinear: 30 clips, sample 'bike-packing' has 6 masks


In [9]:
feat_dir_3d  = os.path.join(FEAT_CACHE_DIR, "anyup3d")
feat_dir_bil = os.path.join(FEAT_CACHE_DIR, "bilinear")

print(f"FEAT_CACHE_DIR: {FEAT_CACHE_DIR}")
print(f"anyup3d:  {len(os.listdir(feat_dir_3d))} files — {os.listdir(feat_dir_3d)[:3]}")
print(f"bilinear: {len(os.listdir(feat_dir_bil))} files — {os.listdir(feat_dir_bil)[:3]}")

print(f"\nSWIN_CACHE_DIR: {SWIN_CACHE_DIR}")
print(f"Swin cache exists: {os.path.isdir(SWIN_CACHE_DIR)}")
if os.path.isdir(SWIN_CACHE_DIR):
    swin_files = os.listdir(SWIN_CACHE_DIR)
    print(f"Swin files: {len(swin_files)} — first 3: {swin_files[:3]}")
    print(f"val_clips[0]: {val_clips[0]}")
    print(f"Expected file: {val_clips[0]}.pt")
    print(f"Match: {val_clips[0]+'.pt' in swin_files}")

FEAT_CACHE_DIR: /content/drive/MyDrive/results/exp3/feat_cache
anyup3d:  30 files — ['bike-packing.pt', 'blackswan.pt', 'bmx-trees.pt']
bilinear: 30 files — ['bike-packing.pt', 'blackswan.pt', 'bmx-trees.pt']

SWIN_CACHE_DIR: /content/drive/MyDrive/results/exp2/swin_cache
Swin cache exists: True
Swin files: 59 — first 3: ['bike-packing.pt', 'blackswan.pt', 'bmx-trees.pt']
val_clips[0]: bike-packing
Expected file: bike-packing.pt
Match: True


In [10]:
# Debug Cell 7 — run this before Cell 10
feat_dir_3d  = os.path.join(FEAT_CACHE_DIR, "anyup3d")
feat_dir_bil = os.path.join(FEAT_CACHE_DIR, "bilinear")

print(f"anyup3d feat cache:  {len(os.listdir(feat_dir_3d))} files")
print(f"bilinear feat cache: {len(os.listdir(feat_dir_bil))} files")
print(f"Val clips: {len(val_clips)}")

# Check if sample_indices is returning anything
sample = val_clips[0]
frame_paths = get_frame_paths(sample)
idxs = sample_indices(len(frame_paths))
print(f"Sample clip '{sample}': {len(frame_paths)} frames, idxs={idxs}")

anyup3d feat cache:  30 files
bilinear feat cache: 30 files
Val clips: 30
Sample clip 'bike-packing': 69 frames, idxs=[0, 12, 24, 36, 48, 60]


In [11]:
# ── Cell 8 · J&F evaluation (no external package) ──────────────────────────
#
# J (Jaccard): IoU between predicted and GT mask per frame, averaged.
# F (F-measure): precision/recall of mask boundaries, averaged.
# Both follow the DAVIS 2017 official definitions:
#   Pont-Tuset et al. arXiv:1704.00675, Section 3.

import cv2
import numpy as np

def compute_J(pred_HW, gt_HW):
    """Jaccard (IoU) for one frame. Ignores background (id=0)."""
    obj_ids = np.unique(gt_HW)
    obj_ids = obj_ids[obj_ids > 0]
    if len(obj_ids) == 0:
        return 1.0   # no object in GT — perfect by convention
    j_scores = []
    for obj_id in obj_ids:
        pred_fg = (pred_HW == obj_id)
        gt_fg   = (gt_HW   == obj_id)
        inter   = np.logical_and(pred_fg, gt_fg).sum()
        union   = np.logical_or(pred_fg,  gt_fg).sum()
        j_scores.append(inter / union if union > 0 else 1.0)
    return float(np.mean(j_scores))


def mask_to_boundary(mask_HW, dilation_px=2):
    """Extract boundary pixels via erosion. Standard DAVIS F-measure approach."""
    mask_u8 = mask_HW.astype(np.uint8)
    kernel  = np.ones((dilation_px * 2 + 1, dilation_px * 2 + 1), np.uint8)
    eroded  = cv2.erode(mask_u8, kernel, iterations=1)
    return mask_u8 - eroded   # boundary = mask minus eroded mask


def compute_F(pred_HW, gt_HW, dilation_px=2):
    """Boundary F-measure for one frame."""
    obj_ids = np.unique(gt_HW)
    obj_ids = obj_ids[obj_ids > 0]
    if len(obj_ids) == 0:
        return 1.0
    f_scores = []
    for obj_id in obj_ids:
        pred_b = mask_to_boundary((pred_HW == obj_id).astype(np.uint8), dilation_px)
        gt_b   = mask_to_boundary((gt_HW   == obj_id).astype(np.uint8), dilation_px)
        tp = np.logical_and(pred_b, gt_b).sum()
        p  = tp / (pred_b.sum() + 1e-8)
        r  = tp / (gt_b.sum()   + 1e-8)
        f  = 2 * p * r / (p + r + 1e-8)
        f_scores.append(float(f))
    return float(np.mean(f_scores))


def evaluate_condition(condition):
    mask_cond_dir = os.path.join(MASK_DIR, condition)
    J_all, F_all  = [], []

    for clip in tqdm(val_clips, desc=condition):
        clip_mask_dir = os.path.join(mask_cond_dir, clip)
        if not os.path.isdir(clip_mask_dir):
            continue

        frame_paths = get_frame_paths(clip)
        ann_paths   = get_ann_paths(clip)
        idxs = sample_indices(len(frame_paths))

        for t, src_i in enumerate(idxs):
            stem = Path(frame_paths[src_i]).stem
            pred_path = os.path.join(clip_mask_dir, f"{stem}.png")
            if not os.path.exists(pred_path):
                continue

            # Load predicted mask and GT at original annotation resolution
            gt_ann_i = src_i if src_i < len(ann_paths) else len(ann_paths) - 1
            gt   = load_ann(ann_paths[gt_ann_i])                    # (H, W) uint8
            pred = np.array(Image.open(pred_path))                   # (H, W) uint8
            if pred.shape != gt.shape:
                pred = cv2.resize(pred, (gt.shape[1], gt.shape[0]),
                                  interpolation=cv2.INTER_NEAREST)

            J_all.append(compute_J(pred, gt))
            F_all.append(compute_F(pred, gt))

    J_mean = float(np.mean(J_all)) if J_all else 0.0
    F_mean = float(np.mean(F_all)) if F_all else 0.0
    return J_mean, F_mean


results = {}
for condition in ["anyup3d", "bilinear"]:
    J, F = evaluate_condition(condition)
    results[condition] = {"J": J, "F": F, "JF": (J + F) / 2.0}
    print(f"  {condition:<10}  J={J:.3f}  F={F:.3f}  J&F={(J+F)/2:.3f}")

anyup3d: 100%|██████████| 30/30 [00:04<00:00,  6.28it/s]


  anyup3d     J=0.240  F=0.048  J&F=0.144


bilinear: 100%|██████████| 30/30 [00:06<00:00,  4.97it/s]

  bilinear    J=0.201  F=0.034  J&F=0.117


In [12]:
# ── Cell 9 · Results table ──────────────────────────────────────────────────
#
# This experiment runs each condition once (no random seed — the propagation
# algorithm is fully deterministic given fixed features and frozen models).
# Statistical comparison across seeds is not applicable here.
# The result is a single clean comparison: bilinear vs AnyUp3D on the same features.

import csv

print("\n── Table: DAVIS 2017 val — Label Propagation ───────────")
print(f"{'Method':<20} {'J':>8} {'F':>8} {'J&F':>8}")
print("-" * 46)
for cond in ["bilinear", "anyup3d"]:
    r = results[cond]
    print(f"{cond:<20} {r['J']:>8.3f} {r['F']:>8.3f} {r['JF']:>8.3f}")
print("-" * 46)

delta_JF = results["anyup3d"]["JF"] - results["bilinear"]["JF"]
delta_J  = results["anyup3d"]["J"]  - results["bilinear"]["J"]
delta_F  = results["anyup3d"]["F"]  - results["bilinear"]["F"]
print(f"{'Delta (3D - bilinear)':<20} {delta_J:>+8.3f} {delta_F:>+8.3f} {delta_JF:>+8.3f}")

if delta_JF > 0:
    print(f"\nAnyUp3D improves J&F by {delta_JF:.3f} points over bilinear.")
else:
    print(f"\nAnyUp3D is {abs(delta_JF):.3f} J&F points below bilinear — "
          "check checkpoint quality or consider more training steps.")

# Save CSV
csv_path = os.path.join(OUTPUT_DIR, "jf_results.csv")
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["method", "J", "F", "JF"])
    for cond in ["bilinear", "anyup3d"]:
        r = results[cond]
        w.writerow([cond, f"{r['J']:.4f}", f"{r['F']:.4f}", f"{r['JF']:.4f}"])
    w.writerow(["delta", f"{delta_J:.4f}", f"{delta_F:.4f}", f"{delta_JF:.4f}"])
print(f"Saved → {csv_path}")


── Table: DAVIS 2017 val — Label Propagation ───────────
Method                      J        F      J&F
----------------------------------------------
bilinear                0.201    0.034    0.117
anyup3d                 0.240    0.048    0.144
----------------------------------------------
Delta (3D - bilinear)   +0.039   +0.015   +0.027

AnyUp3D improves J&F by 0.027 points over bilinear.
Saved → /content/drive/MyDrive/results/exp3/jf_results.csv


In [13]:
# ── Cell 10 · Visual outputs ────────────────────────────────────────────────
#
# (a) Side-by-side stills: N_VISUAL_CLIPS clips × 4 columns
#     [original | GT mask | bilinear propagated | AnyUp3D propagated]
# (b) One full comparison video for the first visual clip.

# Color palette for object IDs (DAVIS standard colors)
PALETTE = np.array([
    [0,   0,   0  ],   # 0: background — black
    [255, 0,   0  ],   # 1: object 1   — red
    [0,   255, 0  ],   # 2: object 2   — green
    [0,   0,   255],   # 3: object 3   — blue
    [255, 255, 0  ],   # 4: object 4   — yellow
    [0,   255, 255],   # 5: object 5   — cyan
], dtype=np.uint8)

def colorize_mask(mask_HW):
    """Integer mask (H,W) → RGB (H,W,3) using PALETTE."""
    ids = np.clip(mask_HW, 0, len(PALETTE) - 1)
    return PALETTE[ids]

def overlay(frame_HWC, mask_HW, alpha=0.55):
    """Blend colorized mask onto RGB frame."""
    color = colorize_mask(mask_HW).astype(np.float32)
    fg = mask_HW > 0
    out = frame_HWC.copy().astype(np.float32)
    out[fg] = out[fg] * (1 - alpha) + color[fg] * alpha
    return np.clip(out, 0, 255).astype(np.uint8)

def load_frame_rgb(path, size):
    return np.array(
        Image.open(path).convert("RGB").resize((size, size), Image.BILINEAR)
    )

def load_mask_png(path, size):
    """Load saved palette PNG → integer numpy (size, size)."""
    m = np.array(Image.open(path))
    return cv2.resize(m, (size, size), interpolation=cv2.INTER_NEAREST).astype(np.int32)

def load_gt_mask(ann_path, size):
    m = load_ann(ann_path)
    return cv2.resize(m, (size, size), interpolation=cv2.INTER_NEAREST).astype(np.int32)


# Pick clips with masks for both conditions
vis_clips = [
    c for c in val_clips
    if (os.path.exists(os.path.join(MASK_DIR, "anyup3d",  c)) and
        os.path.exists(os.path.join(MASK_DIR, "bilinear", c)))
][:N_VISUAL_CLIPS]   # ← N_VISUAL_CLIPS controls figure size

S = 224   # ← display size for figure panels (independent of PROP_SIZE)

print(f"vis_clips: {vis_clips}")
print(f"anyup3d mask dir exists: {os.path.isdir(os.path.join(MASK_DIR, 'anyup3d'))}")
print(f"bilinear mask dir exists: {os.path.isdir(os.path.join(MASK_DIR, 'bilinear'))}")

# List what's actually in the mask dirs
for cond in ["anyup3d", "bilinear"]:
    d = os.path.join(MASK_DIR, cond)
    if os.path.isdir(d):
        clips_found = os.listdir(d)
        print(f"  {cond}: {len(clips_found)} clip folders — first 5: {clips_found[:5]}")
    else:
        print(f"  {cond}: directory does not exist")

# ── (a) Side-by-side stills ──────────────────────────────────────────────────
fig, axes = plt.subplots(
    len(vis_clips), 4,
    figsize=(16, 3.8 * len(vis_clips))
)
if len(vis_clips) == 1:
    axes = [axes]

col_titles = ["Original", "GT mask", "Bilinear", "AnyUp3D"]

for row, clip in enumerate(vis_clips):
    frame_paths = get_frame_paths(clip)
    ann_paths   = get_ann_paths(clip)
    idxs = sample_indices(len(frame_paths))
    mid  = len(idxs) // 2   # middle frame
    src_i = idxs[mid]
    stem  = Path(frame_paths[src_i]).stem

    frame = load_frame_rgb(frame_paths[src_i], S)

    # GT — use the sampled annotation frame
    gt_ann_i = src_i if src_i < len(ann_paths) else len(ann_paths) - 1
    gt_mask  = load_gt_mask(ann_paths[gt_ann_i], S)

    mask_bil_path = os.path.join(MASK_DIR, "bilinear", clip, f"{stem}.png")
    mask_3d_path  = os.path.join(MASK_DIR, "anyup3d",  clip, f"{stem}.png")

    panels = [frame, overlay(frame, gt_mask, alpha=0.6)]
    for path in [mask_bil_path, mask_3d_path]:
        if os.path.exists(path):
            m = load_mask_png(path, S)
            panels.append(overlay(frame, m))
        else:
            panels.append(np.zeros((S, S, 3), dtype=np.uint8))

    for col, (ax, panel) in enumerate(zip(axes[row], panels)):
        ax.imshow(panel)
        ax.axis("off")
        if row == 0:
            ax.set_title(col_titles[col], fontsize=11, fontweight="bold")
    axes[row][0].set_ylabel(clip, fontsize=9)

plt.tight_layout()
stills_path = os.path.join(STILLS_DIR, "side_by_side.png")
plt.savefig(stills_path, dpi=150, bbox_inches="tight")
plt.close()
print(f"Stills saved → {stills_path}")


# ── (b) Comparison video ─────────────────────────────────────────────────────
best_clip   = vis_clips[0]
frame_paths = get_frame_paths(best_clip)
ann_paths   = get_ann_paths(best_clip)
idxs = sample_indices(len(frame_paths))

video_path = os.path.join(VIDEO_DIR, f"{best_clip}_comparison.mp4")
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
# 4 panels: original | GT | bilinear | AnyUp3D
writer = cv2.VideoWriter(video_path, fourcc, VISUAL_FPS, (S * 4, S))

for t, src_i in enumerate(idxs):
    stem  = Path(frame_paths[src_i]).stem
    frame = load_frame_rgb(frame_paths[src_i], S)

    gt_ann_i = src_i if src_i < len(ann_paths) else len(ann_paths) - 1
    gt_mask  = load_gt_mask(ann_paths[gt_ann_i], S)
    p_gt     = overlay(frame, gt_mask, alpha=0.6)

    mask_bil_path = os.path.join(MASK_DIR, "bilinear", best_clip, f"{stem}.png")
    mask_3d_path  = os.path.join(MASK_DIR, "anyup3d",  best_clip, f"{stem}.png")

    p_bil = overlay(frame, load_mask_png(mask_bil_path, S)) \
            if os.path.exists(mask_bil_path) else frame.copy()
    p_3d  = overlay(frame, load_mask_png(mask_3d_path,  S)) \
            if os.path.exists(mask_3d_path)  else frame.copy()

    row_frame = np.concatenate([frame, p_gt, p_bil, p_3d], axis=1)  # (S, 4S, 3)
    writer.write(cv2.cvtColor(row_frame, cv2.COLOR_RGB2BGR))

writer.release()
print(f"Video saved → {video_path}")
print("\n── Experiment 3 complete. ──")

vis_clips: ['bike-packing', 'blackswan', 'bmx-trees', 'breakdance']
anyup3d mask dir exists: True
bilinear mask dir exists: True
  anyup3d: 30 clip folders — first 5: ['bike-packing', 'blackswan', 'bmx-trees', 'breakdance', 'camel']
  bilinear: 30 clip folders — first 5: ['bike-packing', 'blackswan', 'bmx-trees', 'breakdance', 'camel']
Stills saved → /content/drive/MyDrive/results/exp3/stills/side_by_side.png
Video saved → /content/drive/MyDrive/results/exp3/videos/bike-packing_comparison.mp4

── Experiment 3 complete. ──
